# Build your First ML Model

This notebook walks through an end-to-end example of how to download, 
preprocess, train, deploy, and serve a machine learning model using 
Kubeflow Pipelines, MLflow and KServe.

The dataset used is the Wine Quality dataset, and the model predicts 
wine quality using a linear regression approach.

## Table of Contents
1. [Install Dependencies](#Install-Dependencies)
2. [Import Libraries](#Import-libraries)
3. [Kubeflow Pipeline Components](#Kubeflow-Pipeline-Components)
    - [Data Gathering Component](#data-gathering-component)
    - [Preprocessing Component](#preprocessing-component)
    - [Training Component](#training-component)
    - [Deploy Component](#deploy-component)
4. [Kubeflow Pipeline Outline](#Kubeflow-Pipeline-Outline)
5. [Run the Pipeline](#Run-the-Pipeline)
6. [Wait for the Pipeline to Complete](#Wait-for-the-Pipeline-Run-to-Complete)
7. [Test the Deployed Model](#Test-the-Deployed-Model)
8. [Cluster Cleanup](#Cluster-Cleanup)

## Install Dependencies

The following libraries are required to run this notebook:
- **kfp**: define and run Kubeflow Pipelines components
- **mlflow**: track experiments and register the trained model
- **kserve**: deploy the model as a REST inference service
- **tenacity**: handle retries for async operations

In [ ]:
!pip install kfp==2.5.0 mlflow==2.15.1 kserve==0.13.1 tenacity

## Import libraries 

Import the necessary libraries that will be used throughout the notebook.

In [2]:
import kfp
import mlflow
import os
import requests

from kfp.dsl import Input, Model, component
from kfp.dsl import InputPath, OutputPath, pipeline, component
from kserve import KServeClient
from mlflow.tracking import MlflowClient
from tenacity import retry, stop_after_attempt, wait_exponential

## Kubeflow Pipeline Components

Components are the basic building blocks of a Kubeflow Pipeline. They contain all the necessary elements for that specific container. The base image the container will use, dependencies that need to be installed and the code they will execute. 

This allows the minimum required dependencies for the container to run effectively.

The `@component` decorator is used to define these building blocks.

You can read more about Kubeflow Pipelines in the [official documentation](https://www.kubeflow.org/docs/components/pipelines/overview/)

### Data Gathering Component


This container will be in charge of downloading the dataset used for training. 

Note that `requests` and `pandas` are imported directly inside the component. Those are the only dependencies needed for the dataset loading container.

Note that only an `OutputPath` parameter is defined, since the component is in charge of downloading the data with a URL, no need for an `InputPath`.

In [ ]:
@component(
    base_image="python:3.11",
    packages_to_install=["requests==2.32.3", "pandas==2.2.2"]
)
def download_dataset(url: str, dataset_path: OutputPath('Dataset')) -> None:
    import requests
    import pandas as pd

    response = requests.get(url)
    response.raise_for_status()

    from io import StringIO
    dataset = pd.read_csv(StringIO(response.text), header=0, sep=";")

    dataset.to_csv(dataset_path, index=False)

### Preprocessing Component

Now, we define the `preprocess_dataset` component. 

In this case, we have both `InputPath` and `OutputPath` defined:

`dataset: InputPath('Dataset')`: We are taking the dataset from previous component as the input for this preprocess step.
`output_file: OutputPath('Dataset')`: Once the dataset is preprocessed, we are going to output it in a standardized format ready for training.

**Note**: Parquet is a columnar file format optimized for data storage, with smaller sizes compared to CSV.

In [4]:
@component(
    base_image="python:3.11",
    packages_to_install=["pandas==2.2.2", "pyarrow==15.0.2"]
)
def preprocess_dataset(dataset: InputPath('Dataset'), output_file: OutputPath('Dataset')) -> None:
    import pandas as pd
    
    df = pd.read_csv(dataset, header=0)
    df.columns = [c.lower().replace(" ", "_") for c in df.columns]
    df.to_parquet(output_file)

### Training Component

We define the component in charge of training the model with our preprocessed dataset.

#### Steps

1. We read our Parquet file imported as our `InputPath` file.
2. We define the target variable to train on, `quality` in this case.
3. Split the data into train and test with `train_test_split`: we leave 25% of the data for testing purposes.
4. `mlflow.sklearn.autolog()` will log metrics, parameters and the model automatically when running `mlflow.start_run()`. You can learn more about `mlflow.autolog()` in the [official documentation](https://mlflow.org/docs/latest/ml/tracking/autolog)
5. The linear regressor is created and then trained on the training set. The Machine Learning algorithm is `ElasticNet`, a linear regression model with L1 and L2 regularization. Learn more about it in the [official documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html)
6. `model_uri` will be a parameter of the Deploy Component, so it uses the correct model.


In [5]:
@component(
    base_image="python:3.11",
    packages_to_install=["pandas==2.2.2", "scikit-learn==1.5.1", "mlflow==2.15.1", "pyarrow==15.0.2", "boto3==1.34.162"]
)
def train_model(dataset: InputPath('Dataset'), run_name: str, model_name: str) -> str:
    import os
    import mlflow
    import pandas as pd
    from sklearn.linear_model import ElasticNet
    from sklearn.model_selection import train_test_split

    df = pd.read_parquet(dataset)
    
    target_column = "quality"

    train_x, test_x, train_y, test_y = train_test_split(
        df.drop(columns=[target_column]),
        df[target_column], test_size=0.25,
        random_state=42, stratify=df[target_column]
    )

    mlflow.sklearn.autolog()
    
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("author", "kf-testing")
        lr = ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42)
        lr.fit(train_x, train_y)
        mlflow.sklearn.log_model(lr, "model", registered_model_name=model_name)
        
        model_uri = f"{run.info.artifact_uri}/model"
        print(model_uri)
        return model_uri

### Deploy Component

This component takes the `model_uri` returned by the Training Component and deploys it as a REST inference service using KServe.

#### Steps

1. An `InferenceService` object is defined, specifying the model type (`sklearn`), the location of the model (`model_uri`), and the service account with access to S3 where the model is stored.
2. The `InferenceService` is created in the cluster via the KServe client.
3. `tenacity` is used to retry (`@retry` decorator) the readiness check until the service is up, waiting up to 30 attempts before failing. This is due to the async nature of the deploy.
4. Once ready, the inference URL is retrieved and returned. this is the endpoint that will be used to send prediction requests.

**Note**: When creating the Inference Service, Istio sidecar injection is disabled for this service to simplify networking in the cluster (`annotations={"sidecar.istio.io/inject": "false"}`).

In [6]:
@component(
    base_image="python:3.11",
    packages_to_install=["kserve==0.13.1", "kubernetes==26.1.0", "tenacity==9.0.0"]
)
def deploy_model_with_kserve(model_uri: str, isvc_name: str) -> str:
    from kubernetes.client import V1ObjectMeta
    from kserve import (
        constants,
        KServeClient,
        V1beta1InferenceService,
        V1beta1InferenceServiceSpec,
        V1beta1PredictorSpec,
        V1beta1SKLearnSpec,
    )
    from tenacity import retry, wait_exponential, stop_after_attempt

    isvc = V1beta1InferenceService(
        api_version=constants.KSERVE_V1BETA1,
        kind=constants.KSERVE_KIND,
        metadata=V1ObjectMeta(
            name=isvc_name,
            annotations={"sidecar.istio.io/inject": "false"},
        ),
        spec=V1beta1InferenceServiceSpec(
            predictor=V1beta1PredictorSpec(
                service_account_name="kserve-controller-s3",
                sklearn=V1beta1SKLearnSpec(
                    storage_uri=model_uri
                )
            )
        )
    )
    
    client = KServeClient()
    client.create(isvc)

    @retry(
        wait=wait_exponential(multiplier=2, min=1, max=10),
        stop=stop_after_attempt(30),
        reraise=True,
    )
    def assert_isvc_created(client, isvc_name):
        assert client.is_isvc_ready(isvc_name), f"Failed to create Inference Service {isvc_name}."

    assert_isvc_created(client, isvc_name)
    isvc_resp = client.get(isvc_name)
    isvc_url = isvc_resp['status']['address']['url']
    print("Inference URL:", isvc_url)
    
    return isvc_url

# Kubeflow Pipeline Outline

Now that we have defined all the Components, we can assemble them into a 
Kubeflow Pipeline using the `@pipeline` decorator.

The pipeline connects four steps in sequence: data gathering, preprocessing, 
training, and deployment. The output of each step is passed as the input to 
the next.

## Configuration and Pipeline Definition

Before defining the pipeline, we set the names for the MLflow run, the 
registered model, and the KServe inference service.

Credentials for MLflow and S3 are read from environment variables rather 
than hardcoded, which is a security best practice. These need to be set 
in your Kubeflow Notebook environment before running this cell.

In [7]:
ISVC_NAME = "wine-regressor4"
MLFLOW_RUN_NAME = "elastic_net_models"
MLFLOW_MODEL_NAME = "wine-elasticnet"

mlflow_tracking_uri = os.getenv('MLFLOW_TRACKING_URI')
mlflow_s3_endpoint_url = os.getenv('MLFLOW_S3_ENDPOINT_URL')
aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')

@pipeline(name='download-preprocess-train-deploy-pipeline')
def download_preprocess_train_deploy_pipeline(url: str):
    download_task = download_dataset(url=url)
    
    preprocess_task = preprocess_dataset(
        dataset=download_task.outputs['dataset_path']
    )
    
    train_task = train_model(
        dataset=preprocess_task.outputs['output_file'], run_name=MLFLOW_RUN_NAME, model_name=MLFLOW_MODEL_NAME
    ).set_env_variable(name='MLFLOW_TRACKING_URI', value=mlflow_tracking_uri)\
     .set_env_variable(name='MLFLOW_S3_ENDPOINT_URL', value=mlflow_s3_endpoint_url)\
     .set_env_variable(name='AWS_ACCESS_KEY_ID', value=aws_access_key_id)\
     .set_env_variable(name='AWS_SECRET_ACCESS_KEY', value=aws_secret_access_key)
    
    deploy_task = deploy_model_with_kserve(
        model_uri=train_task.output, isvc_name=ISVC_NAME
    ).set_env_variable(name='AWS_SECRET_ACCESS_KEY', value=aws_secret_access_key)

## Run the Pipeline

We connect to the Kubeflow Pipelines API using `kfp.Client()`. It allows 
us to interact with the cluster programmatically.

The pipeline is first compiled into a YAML file. This is the static 
representation of the pipeline that Kubeflow understands. It can also be 
uploaded manually via the Kubeflow UI.

Finally, the pipeline is submitted as a run, passing the dataset URL as 
the argument. 

**Note**: Caching is disabled so that all steps are executed on every 
run, regardless of whether the inputs have changed.

In [8]:
client = kfp.Client()

url = 'https://raw.githubusercontent.com/canonical/kubeflow-examples/main/e2e-wine-kfp-mlflow/winequality-red.csv'

kfp.compiler.Compiler().compile(download_preprocess_train_deploy_pipeline, 'download_preprocess_train_deploy_pipeline.yaml')

run = client.create_run_from_pipeline_func(download_preprocess_train_deploy_pipeline, arguments={'url': url}, enable_caching=False)

/opt/conda/lib/python3.11/site-packages/kfp/client/client.py:159: FutureWarning: This client only works with Kubeflow Pipeline v2.0.0-beta.2 and later versions.
  warnings.warn(


## Wait for the Pipeline Run to Complete

Since the pipeline runs asynchronously in the cluster, this cell polls 
the run status until it reaches a `SUCCEEDED` state.

It retries up to 90 times using an exponential backoff strategy, starting 
with short waits and progressively increasing up to 10 seconds between 
attempts before raising an error if the run didn't succeed.

In [9]:
@retry(
    wait=wait_exponential(multiplier=2, min=1, max=10),
    stop=stop_after_attempt(90),
    reraise=True,
)
def assert_kfp_run_succeeded(client, run_id):
    run = client.get_run(run_id=run_id)
    state = run.state
    assert state == "SUCCEEDED", f"KFP run is in {state} state."

assert_kfp_run_succeeded(client, run.run_id)

## Test the Deployed Model

Once the pipeline has succeeded, we retrieve the inference service URL and send a prediction request to validate that the model is working correctly.

The input consists of two wine samples, each described by 11 feature values. The model returns a predicted quality score for each sample.

In [10]:
kserve_client = KServeClient()

isvc_resp = kserve_client.get(ISVC_NAME)
inference_service_url = isvc_resp['status']['address']['url']
print("Inference URL:", inference_service_url)

input_data = {
    "instances": [
        [7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4],
        [7.8, 0.88, 0.0, 2.6, 0.098, 25.0, 67.0, 0.9968, 3.2, 0.68, 9.8]
    ]
}

response = requests.post(f"{inference_service_url}/v1/models/{ISVC_NAME}:predict", json=input_data)
print(response.text)

Inference URL: http://wine-regressor4.admin.svc.cluster.local
{"predictions":[5.575510497546586,5.527143590500911]}


## Cluster Cleanup

To avoid leaving unused resources running in the cluster, we delete both 
the KServe inference service and the registered model in MLflow.

The deletion of the inference service is also asynchronous, so `tenacity` 
is used again to confirm it has been fully removed before proceeding. The 
check passes either when the service no longer exists or when a "Not Found" 
error is returned by the cluster.


In [11]:
kserve_client.delete(ISVC_NAME)

@retry(
    wait=wait_exponential(multiplier=2, min=1, max=10),
    stop=stop_after_attempt(30),
    reraise=True,
)
def assert_isvc_deleted(kserve_client, isvc_name):
    try:
        isvc = kserve_client.get(isvc_name)
        assert not isvc, f"Failed to delete Inference Service {isvc_name}!"
    except RuntimeError as err:
        assert "Not Found" in str(err), f"Caught unexpected exception: {err}"

assert_isvc_deleted(kserve_client, ISVC_NAME)

client = MlflowClient()
client.delete_registered_model(name=MLFLOW_MODEL_NAME)